# Feedforward on curves, tracker metrics and choosing a tracker: notebook

The car of the chapter (wheelbase 2.5 m, steering at most 26.57°), the test course (10 m straight, 90° left arc of radius 10 m, 10 m straight) and the trackers of RO-071 and RO-072 come from `images/trackkit.py`, copied from those Notes; RO-070's line rule, the feedforward, the metrics and the tighter arc come from `images/ffkit.py`. Trackers run at 20 Hz; the car is stepped every 0.01 s. Every number in the Note is computed here.

In [1]:
import sys
sys.path.insert(0, "images")
from functools import partial
import numpy as np
import trackkit as tk
import ffkit as fk
line = partial(fk.line_rule, ff=False)
line_ff = partial(fk.line_rule, ff=True)
TR = {"pure pursuit": tk.law("pp"), "Stanley": tk.law("stanley"), "rear-wheel feedback": tk.law("rear"),
      "line rule + curve's turn": line_ff}
print("feedforward steering on the 10 m arc:", round(np.degrees(np.arctan(tk.L / 10)), 2), "deg; on a 6 m arc:",
      round(np.degrees(np.arctan(tk.L / 6)), 2), "deg; largest:", round(np.degrees(tk.DELTA_MAX), 2), "deg")
print("sideways acceleration at 5 m/s on the 10 m arc:", 5 ** 2 / 10, "m/s^2")

feedforward steering on the 10 m arc: 14.04 deg; on a 6 m arc: 22.62 deg; largest: 26.57 deg
sideways acceleration at 5 m/s on the 10 m arc: 2.5 m/s^2


## 1. The line rule leaves a steady error on the arc (Note, Sections 1 and 2.1)

In [2]:
for v in (2.0, 5.0):
    for name, law in (("line rule", line), ("line rule + curve's turn", line_ff)):
        r = tk.simulate(law, v=v)
        m = tk.metrics(r)
        s, e = r["s"], r["e"]
        arc = (s >= tk.S1) & (s <= tk.S2)
        end = e[np.argmin(np.abs(s - tk.S2))]
        print(f"{v} m/s {name:26s} within 0.1 m at {m['reach']:.2f} m; arc middle {m['arc_mid']:+.3f} m; end of arc"
              f" {end:+.3f} m; largest on the arc {e[arc][np.argmax(np.abs(e[arc]))]:+.3f} m; RMS {fk.scores(r)['rms']:.3f} m")
    print(f"   by hand -v kappa / K_D = {-v * 0.1 / fk.K_D:+.3f} m; exact steady offset {fk.steady_offset(v=v):+.4f} m")

2.0 m/s line rule                  within 0.1 m at 7.72 m; arc middle -0.354 m; end of arc -0.384 m; largest on the arc -0.384 m; RMS 0.351 m
2.0 m/s line rule + curve's turn   within 0.1 m at 7.72 m; arc middle +0.001 m; end of arc +0.000 m; largest on the arc +0.039 m; RMS 0.265 m
   by hand -v kappa / K_D = -0.400 m; exact steady offset -0.3852 m


5.0 m/s line rule                  within 0.1 m at 7.44 m; arc middle -0.930 m; end of arc -0.938 m; largest on the arc -0.994 m; RMS 0.674 m
5.0 m/s line rule + curve's turn   within 0.1 m at 7.44 m; arc middle -0.031 m; end of arc +0.009 m; largest on the arc -0.104 m; RMS 0.300 m
   by hand -v kappa / K_D = -1.000 m; exact steady offset -0.9161 m


The turn rate on the arc and the steering it means, with no error and with the steady error.

In [3]:
v, kap = 2.0, 0.1
print("turn the arc needs at 2 m/s:", v * kap, "rad/s; steering atan(L * w / v) =", round(np.degrees(np.arctan(tk.L * v * kap / v)), 2), "deg")
e_ss = fk.steady_offset()
print("steady offset", round(e_ss, 4), "m: radius", round(10 - e_ss, 3), "m; turn", round(v / (10 - e_ss), 4), "rad/s = -0.5 e =", round(-0.5 * e_ss, 4))

turn the arc needs at 2 m/s: 0.2 rad/s; steering atan(L * w / v) = 14.04 deg
steady offset -0.3852 m: radius 10.385 m; turn 0.1926 rad/s = -0.5 e = 0.1926


Why RO-070 reports 0.388 m and this notebook 0.384 m for the same line rule on the arc: RO-070 moves the car in straight (Euler) 0.01 s steps, this notebook along exact arcs. The same rule with straight steps:

In [4]:
def simulate_straight_steps(law, v=2.0, start=tk.START, t_max=60.0):
    """tk.simulate without lag, but each 0.01 s step moves the car along a straight line (Euler), as RO-070 does."""
    x, y, th = start
    delta, rows = 0.0, []
    for i in range(int(round(t_max / tk.DT)) + 1):
        if i % tk.HOLD == 0:
            delta = float(np.clip(law(x, y, th, v)["delta"], -tk.DELTA_MAX, tk.DELTA_MAX))
        s, e, the, _ = tk.errors(x, y, th)
        rows.append((s, e))
        if s >= tk.S_END:
            break
        x, y, th = x + v * tk.DT * np.cos(th), y + v * tk.DT * np.sin(th), th + v * np.tan(delta) / tk.L * tk.DT
    return np.array(rows).T
for name, (s, e) in (("exact arcs", (tk.simulate(line)["s"], tk.simulate(line)["e"])), ("straight steps", simulate_straight_steps(line))):
    arc = (s >= tk.S1) & (s <= tk.S2)
    print(f"line rule, 2 m/s, {name:14s}: largest on the arc {e[arc].min():+.4f} m")
print("6 m arc: rear axle radius with the front axle on it", round(np.sqrt(36 - tk.L ** 2), 3), "m")


line rule, 2 m/s, exact arcs    : largest on the arc -0.3844 m
line rule, 2 m/s, straight steps: largest on the arc -0.3882 m
6 m arc: rear axle radius with the front axle on it 5.454 m


## 2. The curvature term already inside the trackers (Note, Section 2.4)

In [5]:
r = tk.simulate(tk.law("rear"), v=2.0, start=(0.0, 0.0, 0.0))
mid = [t for t in r["ticks"] if abs(t["s"] - tk.S_MID) < 0.06][0]
print("rear-wheel feedback mid-arc: ff turn", round(mid["ff"], 4), "rad/s, heading term", round(mid["fb_h"], 5),
      ", lateral term", round(mid["fb_e"], 5), ", steering", round(np.degrees(mid["delta"]), 2), "deg")
st = tk.simulate(tk.law("stanley"), v=2.0, start=(0.0, 0.0, 0.0))
mid = [t for t in st["ticks"] if abs(t["s"] - tk.S_MID) < 0.08][0]
print("Stanley mid-arc: heading term", round(np.degrees(mid["head"]), 2), "deg, cross-track term", round(np.degrees(mid["cross"]), 2),
      "deg, front axle error", round(mid["e"], 4), "m; asin(L / 10) =", round(np.degrees(np.arcsin(tk.L / 10)), 2), "deg")
pp = tk.simulate(tk.law("pp"), v=2.0, start=(0.0, 0.0, 0.0))
i = np.argmin(np.abs(pp["s"] - tk.S_MID))
print("pure pursuit mid-arc steering", round(np.degrees(pp["delta"][i]), 2), "deg, error", round(pp["e"][i], 3), "m")
print("rear axle inside when the front axle is on the arc:", round(10 - np.sqrt(100 - tk.L ** 2), 3), "m (6 m arc:",
      round(6 - np.sqrt(36 - tk.L ** 2), 3), "m)")

rear-wheel feedback mid-arc: ff turn

 0.2 rad/s, heading term 3e-05 , lateral term -0.00028 , steering 14.02 deg
Stanley mid-arc: heading term 13.77 deg, cross-track term 0.04 deg, front axle error -0.0023 m; asin(L / 10) = 14.48 deg


pure pursuit mid-arc steering 14.68 deg, error 0.046 m
rear axle inside when the front axle is on the arc: 0.318 m (6 m arc: 0.546 m)


## 3. The race: the plan's car, starting on the line (Note, Sections 3 and 4.1)

In [6]:
def table(v, tau=0.0, start=(0.0, 0.0, 0.0), extra=None):
    laws = dict(TR, **(extra or {}))
    out = {}
    for name, law in laws.items():
        r = tk.simulate(law, v=v, start=start, tau=tau)
        m = fk.scores(r)
        out[name] = (m, r)
        print(f"  {name:26s} RMS {m['rms']:.3f} m  peak {m['peak']:.3f} m  steering travel {m['travel']:6.1f} deg"
              f"  largest steering {np.degrees(np.abs(r['delta']).max()):5.1f} deg")
    return out
print("2 m/s, plan's car"); t2 = table(2.0, extra={"line rule alone": line})
r = t2["Stanley"][1]
print("Stanley at its own front axle: RMS", round(np.sqrt(np.mean(r["ef"] ** 2)), 3), "m, peak", round(np.abs(r["ef"]).max(), 3), "m")
r = t2["pure pursuit"][1]
print("pure pursuit run: rows", len(r["e"]), "(one per 0.01 s); RMS by the formula", round(np.sqrt(np.sum(r["e"] ** 2) / len(r["e"])), 4),
      "; steering starts rising at s =", round(r["s"][np.argmax(np.abs(r["delta"]) > 1e-3)], 2), "m")
print("RMS checks: constant 0.1 m ->", 0.1, "; 0.1 m half the time ->", round(np.sqrt(0.5 * 0.01), 4))

2 m/s, plan's car


  pure pursuit               RMS 0.086 m  peak 0.167 m  steering travel   30.1 deg  largest steering  14.7 deg
  Stanley                    RMS 0.189 m  peak 0.316 m  steering travel   28.8 deg  largest steering  14.4 deg


  rear-wheel feedback        RMS 0.002 m  peak 0.007 m  steering travel   33.8 deg  largest steering  15.4 deg


  line rule + curve's turn   RMS 0.003 m  peak 0.007 m  steering travel   33.5 deg  largest steering  15.4 deg


  line rule alone            RMS 0.234 m  peak 0.384 m  steering travel   34.8 deg  largest steering  15.6 deg
Stanley at its own front axle: RMS 0.003 m, peak 0.005 m
pure pursuit run: rows 1783 (one per 0.01 s); RMS by the formula 0.0859 ; steering starts rising at s = 6.3 m
RMS checks: constant 0.1 m -> 0.1 ; 0.1 m half the time -> 0.0707


The getting-on test from the chapter's start, 1.0 m left of the line (distance to within 0.1 m at the rear axle).

In [7]:
for name, law in TR.items():
    r = tk.simulate(law, v=2.0)
    m = tk.metrics(r); mf = tk.metrics(r, ref="front")
    print(f"  {name:26s} rear axle within 0.1 m at {m['reach']:.2f} m; front axle at {mf['reach']:.2f} m; RMS {fk.scores(r)['rms']:.3f} m")

  pure pursuit               rear axle within 0.1 m at 7.36 m; front axle at 6.53 m; RMS 0.297 m
  Stanley                    rear axle within 0.1 m at 27.94 m; front axle at 9.33 m; RMS 0.371 m


  rear-wheel feedback        rear axle within 0.1 m at 4.09 m; front axle at 3.84 m; RMS 0.227 m
  line rule + curve's turn   rear axle within 0.1 m at 7.72 m; front axle at 5.87 m; RMS 0.265 m


## 4. Speed (Note, Section 4.2)

In [8]:
print("5 m/s, plan's car"); t5 = table(5.0, extra={"line rule alone": line})
r = t5["Stanley"][1]
print("Stanley at its own front axle, 5 m/s: RMS", round(np.sqrt(np.mean(r["ef"] ** 2)), 3), "m, peak", round(np.abs(r["ef"]).max(), 3), "m")
print("2 m/s, steering lag 0.4 s"); t2l = table(2.0, tau=0.4)
print("5 m/s, steering lag 0.4 s"); t5l = table(5.0, tau=0.4)
print("distance driven during the lag:", 2 * 0.4, "m at 2 m/s,", 5 * 0.4, "m at 5 m/s")

5 m/s, plan's car


  pure pursuit               RMS 0.082 m  peak 0.161 m  steering travel   30.1 deg  largest steering  14.7 deg


  Stanley                    RMS 0.185 m  peak 0.310 m  steering travel   28.7 deg  largest steering  14.4 deg
  rear-wheel feedback        RMS 0.001 m  peak 0.003 m  steering travel   29.4 deg  largest steering  14.0 deg


  line rule + curve's turn   RMS 0.002 m  peak 0.006 m  steering travel   28.6 deg  largest steering  14.0 deg


  line rule alone            RMS 0.600 m  peak 0.997 m  steering travel   37.2 deg  largest steering  16.4 deg
Stanley at its own front axle, 5 m/s: RMS 0.01 m, peak 0.017 m
2 m/s, steering lag 0.4 s


  pure pursuit               RMS 0.042 m  peak 0.098 m  steering travel   31.9 deg  largest steering  14.9 deg
  Stanley                    RMS 0.177 m  peak 0.306 m  steering travel   30.7 deg  largest steering  14.4 deg


  rear-wheel feedback        RMS 0.028 m  peak 0.082 m  steering travel   96.1 deg  largest steering  20.6 deg
  line rule + curve's turn   RMS 0.033 m  peak 0.089 m  steering travel   71.8 deg  largest steering  19.4 deg
5 m/s, steering lag 0.4 s


  pure pursuit               RMS 0.080 m  peak 0.180 m  steering travel   49.6 deg  largest steering  17.5 deg
  Stanley                    RMS 0.170 m  peak 0.312 m  steering travel   55.1 deg  largest steering  17.0 deg
  rear-wheel feedback        RMS 0.235 m  peak 0.595 m  steering travel  202.6 deg  largest steering  24.5 deg


  line rule + curve's turn   RMS 0.203 m  peak 0.414 m  steering travel   74.8 deg  largest steering  20.1 deg
distance driven during the lag: 0.8 m at 2 m/s, 2.0 m at 5 m/s


The line rule's damping against speed: on a straight, small errors, e'' + K_H e' + v K_D e = 0.

In [9]:
for v in (2.0, 5.0):
    wn = np.sqrt(v * fk.K_D)
    print(f"v = {v}: natural frequency {wn:.3f} rad/s, damping ratio {fk.K_H / (2 * wn):.3f}")
print("rear-wheel feedback, gains halved and quartered at 5 m/s with the lag:")
for kth, ke in ((1.0, 0.5), (0.5, 0.125)):
    r = tk.simulate(tk.law("rear", kth=kth, ke=ke), v=5.0, start=(0.0, 0.0, 0.0), tau=0.4)
    m = fk.scores(r)
    print(f"  k_th {kth}, k_e {ke}: RMS {m['rms']:.3f} m, peak {m['peak']:.3f} m, travel {m['travel']:.1f} deg")

v = 2.0: natural frequency 1.000 rad/s, damping ratio 1.000
v = 5.0: natural frequency 1.581 rad/s, damping ratio 0.632
rear-wheel feedback, gains halved and quartered at 5 m/s with the lag:
  k_th 1.0, k_e 0.5: RMS 0.235 m, peak 0.595 m, travel 202.6 deg


  k_th 0.5, k_e 0.125: RMS 0.171 m, peak 0.367 m, travel 89.8 deg


## 5. Curvature: the same course with a 6 m arc (Note, Section 4.3)

In [10]:
old = fk.set_arc(6.0)
print("2 m/s, plan's car, 6 m arc"); t6 = table(2.0, extra={"line rule alone": line})
fk.set_arc(old)

2 m/s, plan's car, 6 m arc


  pure pursuit               RMS 0.144 m  peak 0.280 m  steering travel   48.3 deg  largest steering  23.1 deg
  Stanley                    RMS 0.227 m  peak 0.513 m  steering travel   47.7 deg  largest steering  23.9 deg


  rear-wheel feedback        RMS 0.003 m  peak 0.011 m  steering travel   53.3 deg  largest steering  24.7 deg
  line rule + curve's turn   RMS 0.004 m  peak 0.012 m  steering travel   52.7 deg  largest steering  24.6 deg


  line rule alone            RMS 0.276 m  peak 0.592 m  steering travel   54.6 deg  largest steering  24.4 deg


6.0